# Training Models

This notebook demonstrates how to train the three channel estimation models:
1. **Linear** - Simple learned linear estimator baseline
2. **FortiTran** - Transformer-based parameter-free channel estimator
3. **AdaFortiTran** - Adaptive transformer with channel condition awareness

**Prerequisites:** Run `01_data_setup.ipynb` first to prepare the training data.

In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.4.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3060


## 1. Configuration

Define paths and training parameters.

In [2]:
# Paths
TRAIN_DIR = PROJECT_ROOT / "data" / "train"
VAL_DIR = PROJECT_ROOT / "data" / "val"
SYSTEM_CONFIG = PROJECT_ROOT / "config" / "system_config.yaml"
ADAFORTITRAN_CONFIG = PROJECT_ROOT / "config" / "adafortitran.yaml"
FORTITRAN_CONFIG = PROJECT_ROOT / "config" / "fortitran.yaml"

# Verify data exists
train_count = len(list(TRAIN_DIR.glob("*.mat")))
val_count = len(list(VAL_DIR.glob("*.mat")))
print(f"Training samples: {train_count}")
print(f"Validation samples: {val_count}")

if train_count == 0 or val_count == 0:
    print("No data found! Please run 01_data_setup.ipynb first.")

Training samples: 800
Validation samples: 200


## 2. Training Parameters

Adjust these parameters based on your hardware and requirements.

In [4]:
# Training hyperparameters (feel free to modify)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 4

# Device selection
DEVICE = "cuda"

## 3. Train Models

Choose which model to train by running the corresponding cell.

### Option A: Train Linear Model (Fastest - Good for Testing Pipeline)

In [5]:
import subprocess
import os

# Change to project root for running the training script
os.chdir(PROJECT_ROOT)

# Set PYTHONPATH so src module can be found
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

# Train Linear model
cmd = [
    sys.executable, "src/train.py",
    "--model_name", "linear",
    "--system_config_path", str(SYSTEM_CONFIG),
    "--train_set", str(TRAIN_DIR),
    "--val_set", str(VAL_DIR),
    "--exp_id", "linear_demo",
    "--batch_size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--max_epoch", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--num_workers", str(NUM_WORKERS),
    "--device", DEVICE
]

print("Running command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

result = subprocess.run(cmd, capture_output=False, env=env)
print(f"\nTraining finished with exit code: {result.returncode}")

Running command:
/home/berkay/miniconda3/envs/custom_ce/bin/python src/train.py --model_name linear --system_config_path /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml --train_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/train --val_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/val --exp_id linear_demo --batch_size 64 --lr 0.001 --max_epoch 10 --patience 3 --num_workers 4 --device cuda


2026-01-09 16:49:57,087 - __main__ - INFO - Starting OFDM channel estimation model training
2026-01-09 16:49:57,087 - __main__ - INFO - Model: linear
2026-01-09 16:49:57,087 - __main__ - INFO - Device: cuda
2026-01-09 16:49:57,087 - __main__ - INFO - System config: /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml
2026-01-09 16:49:57,087 - __main__ - INFO - Model config: Not applicable for linear model
2026-01-09 16:49:57,087 - __main__ - INFO - Experiment ID: linear_demo
2026-01-09 16:49:57,087 - __main__ - INFO - Loading config

Epoch 10/10 - Train Loss: 0.0559, Val Loss: 0.0581: 100%|██████████| 10/10 [00:01<00:00,  6.14it/s]


2026-01-09 16:49:59,278 - src.main.trainer - INFO - Checkpoint saved to runs/linear_linear_demo/periodic/checkpoint_epoch_9.pt
2026-01-09 16:49:59,285 - __main__ - INFO - Training completed successfully

Training finished with exit code: 0


### Option B: Train FortiTran Model

In [6]:
# Train FortiTran model
os.chdir(PROJECT_ROOT)

# Set PYTHONPATH so src module can be found
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

cmd = [
    sys.executable, "src/train.py",
    "--model_name", "fortitran",
    "--system_config_path", str(SYSTEM_CONFIG),
    "--model_config_path", str(FORTITRAN_CONFIG),
    "--train_set", str(TRAIN_DIR),
    "--val_set", str(VAL_DIR),
    "--exp_id", "fortitran_demo",
    "--batch_size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--max_epoch", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--num_workers", str(NUM_WORKERS),
    "--device", DEVICE
]

print("Running command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

result = subprocess.run(cmd, capture_output=False, env=env)
print(f"\nTraining finished with exit code: {result.returncode}")

Running command:
/home/berkay/miniconda3/envs/custom_ce/bin/python src/train.py --model_name fortitran --system_config_path /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml --model_config_path /home/berkay/Desktop/research/2026/AdaFortiTran/config/fortitran.yaml --train_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/train --val_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/val --exp_id fortitran_demo --batch_size 64 --lr 0.001 --max_epoch 10 --patience 3 --num_workers 4 --device cuda


2026-01-09 16:50:17,809 - __main__ - INFO - Starting OFDM channel estimation model training
2026-01-09 16:50:17,809 - __main__ - INFO - Model: fortitran
2026-01-09 16:50:17,809 - __main__ - INFO - Device: cuda
2026-01-09 16:50:17,809 - __main__ - INFO - System config: /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml
2026-01-09 16:50:17,809 - __main__ - INFO - Model config: /home/berkay/Desktop/research/2026/AdaFortiTran/config/fortitr

Epoch 9/10 - Train Loss: 0.1384, Val Loss: 0.1303:  90%|█████████ | 9/10 [00:22<00:02,  2.46s/it]

2026-01-09 16:50:43,143 - src.main.trainer - INFO - Checkpoint saved to runs/fortitran_fortitran_demo/periodic/checkpoint_epoch_9.pt
2026-01-09 16:50:43,146 - __main__ - INFO - Training completed successfully


Epoch 10/10 - Train Loss: 0.1300, Val Loss: 0.1220: 100%|██████████| 10/10 [00:24<00:00,  2.49s/it]



Training finished with exit code: 0


### Option C: Train AdaFortiTran Model (Recommended)

In [7]:
# Train AdaFortiTran model
os.chdir(PROJECT_ROOT)

# Set PYTHONPATH so src module can be found
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

cmd = [
    sys.executable, "src/train.py",
    "--model_name", "adafortitran",
    "--system_config_path", str(SYSTEM_CONFIG),
    "--model_config_path", str(ADAFORTITRAN_CONFIG),
    "--train_set", str(TRAIN_DIR),
    "--val_set", str(VAL_DIR),
    "--exp_id", "adafortitran_demo",
    "--batch_size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--max_epoch", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--num_workers", str(NUM_WORKERS),
    "--device", DEVICE
]

print("Running command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

result = subprocess.run(cmd, capture_output=False, env=env)
print(f"\nTraining finished with exit code: {result.returncode}")

Running command:
/home/berkay/miniconda3/envs/custom_ce/bin/python src/train.py --model_name adafortitran --system_config_path /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml --model_config_path /home/berkay/Desktop/research/2026/AdaFortiTran/config/adafortitran.yaml --train_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/train --val_set /home/berkay/Desktop/research/2026/AdaFortiTran/data/val --exp_id adafortitran_demo --batch_size 64 --lr 0.001 --max_epoch 10 --patience 3 --num_workers 4 --device cuda


2026-01-09 16:50:51,942 - __main__ - INFO - Starting OFDM channel estimation model training
2026-01-09 16:50:51,942 - __main__ - INFO - Model: adafortitran
2026-01-09 16:50:51,942 - __main__ - INFO - Device: cuda
2026-01-09 16:50:51,942 - __main__ - INFO - System config: /home/berkay/Desktop/research/2026/AdaFortiTran/config/system_config.yaml
2026-01-09 16:50:51,942 - __main__ - INFO - Model config: /home/berkay/Desktop/research/2026/AdaFortiTran/co

Epoch 9/10 - Train Loss: 0.1568, Val Loss: 0.1465:  90%|█████████ | 9/10 [00:22<00:02,  2.48s/it]

2026-01-09 16:51:17,280 - src.main.trainer - INFO - Checkpoint saved to runs/adafortitran_adafortitran_demo/periodic/checkpoint_epoch_9.pt
2026-01-09 16:51:17,282 - __main__ - INFO - Training completed successfully


Epoch 10/10 - Train Loss: 0.1480, Val Loss: 0.1380: 100%|██████████| 10/10 [00:24<00:00,  2.50s/it]



Training finished with exit code: 0


## 4. Check Training Results

After training, checkpoints are saved in the `runs/` directory.

In [9]:
# List available checkpoints
runs_dir = PROJECT_ROOT / "runs"

if runs_dir.exists():
    print("Available experiment runs:")
    print("="*60)
    for exp_dir in sorted(runs_dir.iterdir()):
        if exp_dir.is_dir():
            checkpoints = list(exp_dir.rglob("*.pt"))
            print(f"\n{exp_dir.name}")
            for ckpt in checkpoints[:5]:  # Show up to 5 checkpoints
                rel_path = ckpt.relative_to(PROJECT_ROOT)
                print(f"   └── {rel_path}")
            if len(checkpoints) > 5:
                print(f"   └── ... and {len(checkpoints) - 5} more")
else:
    print("No runs directory found. Train a model first!")

Available experiment runs:

adafortitran_adafortitran_demo
   └── runs/adafortitran_adafortitran_demo/periodic/checkpoint_epoch_9.pt

fortitran_fortitran_demo
   └── runs/fortitran_fortitran_demo/periodic/checkpoint_epoch_9.pt

linear_linear_demo
   └── runs/linear_linear_demo/periodic/checkpoint_epoch_9.pt


## 5. TensorBoard Visualization (Optional)

To visualize training metrics, run the following command in your terminal:

```bash
tensorboard --logdir runs/
```

Then open http://localhost:6006 in your browser.

---

**Next Steps:** Open `03_inference.ipynb` to load a trained model and run inference.